# OpEn in Rust: constrained Rosenbrock minimization

This notebook mirrors the basic Rust example from the OpEn docs and solves a simple optimization problem with `optimization_engine`.

Reference: https://alphaville.github.io/optimization-engine/docs/openrust-basic

We minimize the two-dimensional Rosenbrock function with parameters `a = 1` and `b = 200`, subject to the constraint that the decision variable lies inside the unit Euclidean ball.

In [ ]:
:dep optimization_engine = "=0.11.0"

use optimization_engine::{
    constraints::Ball2,
    panoc::{PANOCCache, PANOCOptimizer},
    Optimizer,
    Problem,
    SolverError,
};

Let us define the cost function and its gradient...

In [ ]:
fn rosenbrock_cost(a: f64, b: f64, u: &[f64]) -> f64 {
    (a - u[0]).powi(2) + b * (u[1] - u[0].powi(2)).powi(2)
}

fn rosenbrock_grad(a: f64, b: f64, u: &[f64], grad: &mut [f64]) {
    grad[0] = 2.0 * u[0] - 2.0 * a - 4.0 * b * u[0] * (-u[0].powi(2) + u[1]);
    grad[1] = b * (-2.0 * u[0].powi(2) + 2.0 * u[1]);
}

and now let us define some constants

In [ ]:
let tolerance :f64= 1e-14;
let a :f64= 1.0;
let b :f64= 200.0;
let problem_size :usize= 2;
let lbfgs_memory_size :usize = 10;
let max_iters :usize = 80;
let mut u = [-1.5, 0.9];
let radius :f64 = 1.0;

We can now solve the optimisation problem...

In [ ]:
{

    let df = move |u: &[f64], grad: &mut [f64]| -> Result<(), SolverError> {
        if a < 0.0 || b < 0.0 {
            Err(SolverError::Cost)
        } else {
            rosenbrock_grad(a, b, u, grad);
            Ok(())
        }
    };

    let f = move |u: &[f64], c: &mut f64| -> Result<(), SolverError> {
        if a < 0.0 || b < 0.0 {
            Err(SolverError::Cost)
        } else {
            *c = rosenbrock_cost(a, b, u);
            Ok(())
        }
    };

    let bounds = Ball2::new(None, radius);
    let problem = Problem::new(&bounds, df, f);
    let mut panoc_cache = PANOCCache::new(problem_size, tolerance, lbfgs_memory_size);
    let mut panoc = PANOCOptimizer::new(problem, &mut panoc_cache).with_max_iter(max_iters);

    let status = panoc.solve(&mut u).unwrap();
    println!("Converged: {}", status.has_converged());
    println!("Iterations: {}", status.iterations());
    println!("Cost: {:.12}", status.cost_value());
    println!("Solution: {:?}", u);
    u
}